# Generate and save Wiener filter (WF) decoders

Three ways to create a `WFDecoder` for EMG (or any continuous feature) cursor control, and save it to the database so it can be selected in the BMI3D task interface:

1. **Fit from training data**: least-squares fit of the filter weights to the velocity of a recorded session (e.g., a manual control or tracking task).
2. **Random initialization**: random weights, e.g., as a naive seed for closed-loop decoder adaptation (CLDA).
3. **Fixed, hand-designed weights**: e.g., groups of channels driving the cursor in each direction.

The Wiener filter estimates the velocity states of the cursor as a linear function of the last `n_taps` bins of features plus an offset,
`[vx, vz] = H * [y_t; y_{t-1}; ...; y_{t-n_taps+1}; 1]`, and the position states integrate the decoded velocity.
The weights of the decoder can be adapted online with the `clda_wf_smoothbatch` feature (`riglib.bmi.clda.WFSmoothbatch`).

Requirements: a version of bmi3d with the Wiener filter decoder (`riglib.bmi.wfdecoder`) importable from this kernel (e.g., this repository first on `sys.path`, or installed in the kernel's environment), and a recent `aopy`. Saving requires the `bmi` system path to be available on this machine (see the BMI3D setup page).

In [1]:
import os
import h5py
import aopy
from aopy.data import db
import matplotlib.pyplot as plt
import numpy as np
import sklearn.metrics
import riglib
from riglib.bmi import train
from riglib.bmi.extractor import rms_emg
from riglib.bmi.state_space_models import StateSpaceEndptVel2D

print('riglib from', riglib.__file__)
print('aopy from', aopy.__file__)
assert hasattr(train, 'rand_WFDecoder'), "this version of bmi3d has no Wiener filter decoder; put the right repository first on sys.path before importing riglib"

/home/aolab/miniconda3/envs/bmi3d/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/aolab/miniconda3/envs/bmi3d/lib/python3.9/site-packages/one/alf/files.py:10: FutureWarning: `one.alf.files` will be removed in version 3.0. Use `one.alf.path` instead.
  warnings.warn(


riglib from /home/aolab/bmi3d/riglib/__init__.py
aopy from /home/aolab/code/analyze/aopy/__init__.py


## Settings

In [2]:
data_dir = '/human-data'
preproc_dir = '/home/aolab/emg_preproc'
db.BMI3D_DBNAME = 'human'

# Decoder settings
binlen = 0.0167  # time between decoder updates [s]; the task calls the decoder at 60 Hz and accumulates features over the bin
n_taps = 1       # number of bins of feature history the filter acts on, including the current one
zscore = True    # normalize the features by their mean and standard deviation before the filter. Recommended: the weights are
                 # then in cursor units per standard deviation of each feature, so the CLDA ridge penalty is meaningful as is.

# State space: [px, py, pz, vx, vy, vz, offset]. The filter estimates vx and vz, and px and pz integrate them.
ssm = StateSpaceEndptVel2D()

# Feature extractor: RMS of each EMG channel over the last win_len seconds
n_channels = 64
extractor_kwargs = dict(channels=list(range(1, n_channels+1)), win_len=0.2)

## Helper functions

In [3]:
def attach_extractor(decoder, units):
    '''
    Tell the decoder which feature extractor to run in the task and which channels to use
    '''
    decoder.extractor_cls = rms_emg
    decoder.extractor_kwargs = dict(extractor_kwargs, channels=[int(c) for c, u in units])
    return decoder

def train_wf_from_data(ssm, units, vel, features, dt, n_taps, shift=1, **kwargs):
    '''
    Wrapper around riglib.bmi.train.train_WFDecoder_abstract

    vel : (nt, 2) x and y cursor velocity
    features : (nt, n_units) features, e.g., EMG amplitude
    shift : the filter is trained to predict the velocity 'shift' bins after the current features
        (1 bin, like riglib.bmi.train.train_WFDecoder, since the decoded velocity moves the cursor in the next bin)
    kwargs : e.g. regularizer (ridge penalty on the weights) or zscore; see train_WFDecoder_abstract
    '''
    units = np.array(units)
    nt = len(vel) - shift
    kin = np.zeros((ssm.n_states, nt))
    kin[ssm.train_inds, :] = vel[shift:].T
    feat = features[:nt].T
    decoder = train.train_WFDecoder_abstract(ssm, kin, feat, units, dt, n_taps=n_taps, **kwargs)
    return attach_extractor(decoder, units)

def evaluate_wf(decoder, features, vel, shift=1, plot=True):
    '''
    Decode the features offline and compare the decoded velocity to the actual velocity (R^2 for x and y)
    '''
    decoder.filt._init_state()
    out = decoder.decode(features.T)
    pred = out[:len(out)-shift, :][:, decoder.ssm.train_inds]
    actual = vel[shift:]
    r2 = sklearn.metrics.r2_score(actual, pred, multioutput='raw_values')
    if plot:
        t = np.arange(len(actual)) * decoder.binlen
        fig, ax = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
        for k, name in enumerate(['x', 'y']):
            ax[k].plot(t, actual[:, k], label='actual', lw=1)
            ax[k].plot(t, pred[:, k], label='decoded', lw=1, alpha=0.8)
            ax[k].set_ylabel(f'{name} velocity')
            ax[k].set_title(f'R$^2$ = {r2[k]:.3f}')
        ax[0].legend(loc='upper right')
        ax[1].set_xlabel('time (s)')
        fig.tight_layout()
    return r2

def plot_wf_weights(decoder, title=''):
    '''
    Plot the filter weights of each lag, and the lag-0 weights of each feature as a vector in velocity space
    '''
    H = np.asarray(decoder.filt.H)
    n_feat, n_taps = decoder.filt.n_features, decoder.filt.n_taps
    inds = list(decoder.ssm.train_inds)
    names = [decoder.states[i] for i in inds]
    fig, ax = plt.subplots(1, n_taps + 1, figsize=(3.5*n_taps + 5, 3.5), constrained_layout=True)
    vmax = np.max(np.abs(H[inds, :-1])) or 1.
    for k in range(n_taps):
        Hk = H[inds, k*n_feat:(k+1)*n_feat]
        im = ax[k].pcolor(Hk, cmap='bwr', vmin=-vmax, vmax=vmax)
        ax[k].set_yticks(np.arange(len(inds)) + .5)
        ax[k].set_yticklabels(names)
        ax[k].set_xlabel('feature')
        ax[k].set_title(f'lag {k} ({k*decoder.binlen*1000:.0f} ms)')
    fig.colorbar(im, ax=ax[:n_taps].tolist(), label='weight', shrink=0.8)
    H0 = H[inds, :n_feat]
    lim = np.max(np.linalg.norm(H0, axis=0)) or 1.
    ax[-1].quiver(np.zeros(n_feat), np.zeros(n_feat), H0[0], H0[1], angles='xy', scale_units='xy', scale=1., width=.005, alpha=.6)
    ax[-1].set(xlim=(-lim, lim), ylim=(-lim, lim), xlabel=names[0], ylabel=names[1], title='lag 0 weight vectors')
    ax[-1].set_aspect('equal')
    fig.suptitle(f'{title} decoder, {n_taps} taps, offset = {np.round(H[inds, -1], 3)}')

## 1. Fit a decoder from training data

Load a recorded session, extract the features the task would have computed online, and fit the filter to the cursor velocity.

In [4]:
entries = db.lookup_sessions(subject='Marios', date=('2026-09-24', '2026-09-25'))
for e in entries:
    print(f"\033[1m{e.id}\033[0;0m {e.subject} on {e.task_name} task ({e.task_desc})\nProject: {e.project}, session: {e.session}\nNotes: {e.notes}")
    print(e.get_raw_files())
    print()

te_id = entries[-1].id  # session to train on
entry = [e for e in entries if e.id == te_id][0]
subject, date = entry.subject, entry.date

6255 Marios on myo tracking task (2d tracking)
Project: test, session: test
Notes: 
{'hdf': 'hdf/mari20260924_01_te6255.hdf', 'emg': 'emg/mari20260924_01_te6255.hdf'}

6256 Marios on myo tracking task (2d tracking)
Project: clda test, session: test
Notes: 
{'hdf': 'hdf/mari20260924_02_te6256.hdf', 'emg': 'emg/mari20260924_02_te6256.hdf'}



In [5]:
!~/code/nightly_script.sh

# Preprocess the raw data, if not done already
entry.preprocess(data_dir, preproc_dir, exclude_sources=['eye'], overwrite=True)

Updating hdf file with metadata for task entry 6256
/human-data/hdf/mari20260924_02_te6256.hdf
processing experiment data...
done!
processing emg data...


In [6]:
# Extract features the same way the task does online. Only the extractor settings and bin length of the decoder
# passed in are used, so a random decoder with the right settings serves as the template
units = [(c, 0) for c in extractor_kwargs['channels']]
template = attach_extractor(train.rand_WFDecoder(ssm, np.array(units), dt=binlen, n_taps=n_taps), units)
neural, samplerate = aopy.data.extract_lfp_features(preproc_dir, subject, te_id, date, template, samplerate=1./binlen, datatype='emg')

# Cursor velocity at the same sample times. aopy (x, y) correspond to bmi3d (x, z)
cursor, _ = aopy.data.get_kinematics(preproc_dir, subject, te_id, date, samplerate)
nt = min(len(neural), len(cursor))
vel = aopy.utils.derivative(np.arange(nt)/samplerate, cursor[:nt, :2], norm=False)
neural_all, neural = neural[:nt], neural[:nt]
print(f'{nt} samples at {samplerate} Hz: features {neural.shape}, velocity {vel.shape}')
print(f'feature amplitude: mean {np.mean(neural):.2e}, std {np.std(neural):.2e}')

Extracting LFP features:   4%|▍         | 1728/43586 [00:24<09:53, 70.56it/s]


KeyboardInterrupt: 

In [ ]:
# Remove noisy channels
bad_ch = aopy.preproc.quality.detect_bad_ch_outliers(neural, thr=0.1, numsd=2.0, debug=True)
good_ch = np.nonzero(~bad_ch)[0]
units = [(int(i+1), 0) for i in good_ch]
neural = neural[:, good_ch]
print(f'{len(units)} channels kept')

In [ ]:
# Choose the number of taps and the regularization on held-out data. The ridge penalty is specified relative to
# the scale of the feature Gram matrix so that the values below mean the same thing whatever the feature units
n_train = int(0.7 * nt)
neural_train, vel_train = neural[:n_train], vel[:n_train]
neural_test, vel_test = neural[n_train:], vel[n_train:]
if zscore:
    gram_scale = n_train
else:
    gram_scale = n_train * np.mean(neural_train**2)

print('R^2 (x, y) on held-out data')
print('n_taps ' + ' '.join(f'{a:>16}' for a in [0, 1e-3, 1e-2, 1e-1]))
for k in [1, 2, 3, 5, 8]:
    row = []
    for alpha in [0, 1e-3, 1e-2, 1e-1]:
        dec = train_wf_from_data(ssm, units, vel_train, neural_train, binlen, k, zscore=zscore, regularizer=alpha*gram_scale)
        r2 = evaluate_wf(dec, neural_test, vel_test, plot=False)
        row.append(f'({r2[0]:.3f}, {r2[1]:.3f})')
    print(f'{k:>6} ' + ' '.join(f'{r:>16}' for r in row))

In [ ]:
# Fit the final decoder on all the data
n_taps = 3
alpha = 1e-2
gram_scale = nt if zscore else nt * np.mean(neural**2)
fit_decoder = train_wf_from_data(ssm, units, vel, neural, binlen, n_taps, zscore=zscore, regularizer=alpha*gram_scale)
plot_wf_weights(fit_decoder, 'fit')
r2 = evaluate_wf(fit_decoder, neural, vel)
print('R^2 (x, y) on the training data:', np.round(r2, 3))

In [ ]:
# Save the decoder to the database, associated with the session it was trained on
db.save_decoder(entry, fit_decoder, 'fit_wf')

## 2. Random initialization

Random weights with zero offset, e.g., as a naive seed for CLDA. The weights are scaled so that a typical feature vector produces a cursor speed of about `target_speed`.
The features are normalized with the statistics of the session loaded above so that the weights are in the same units as a fitted decoder.

In [10]:
target_speed = 5.  # cursor speed [cm/s] produced by a typical (one standard deviation) feature vector
units = [(c, 0) for c in extractor_kwargs['channels']]  # all channels, or reuse the good channels from above
scale = target_speed / np.sqrt(len(units) * n_taps)

np.random.seed(0)
rand_decoder = attach_extractor(train.rand_WFDecoder(ssm, np.array(units), dt=binlen, n_taps=n_taps, scale=scale), units)

# Feature normalization from the session above ('neural_all' has all the channels, in the same order as 'units'). Without a
# session, use a constant guess, e.g., rand_decoder.init_zscore(np.ones(len(units))*1e-5, np.ones(len(units))*1e-5)
rand_decoder.init_zscore(np.mean(neural_all, axis=0), np.std(neural_all, axis=0))

plot_wf_weights(rand_decoder, 'random')
r2 = evaluate_wf(rand_decoder, neural_all, vel)
print('R^2 (x, y) of the random decoder on the session above:', np.round(r2, 3))

NameError: name 'neural_all' is not defined

In [ ]:
# Decoders that are not trained on a session are attached to a "decoder parent" entry, looked up or created by project and session
parents = db.lookup_decoder_parent(project='emg_pilot', session='wf_decoders')
parent = parents[0] if len(parents) > 0 else db.create_decoder_parent('emg_pilot', 'wf_decoders')
db.save_decoder(parent, rand_decoder, 'rand_wf')

## 3. Fixed, hand-designed weights

Specify the weights directly, e.g., groups of channels that drive the cursor in each direction. Here the weights are in cursor units per standard deviation of each (normalized) feature.

In [ ]:
units = [(c, 0) for c in extractor_kwargs['channels']]
n_taps_fixed = 1
gain = 1.  # cm/s per standard deviation of each feature in a group

# rows: hand_vx, hand_vz; columns: channels (index = channel - 1)
H_fixed = np.zeros((2, len(units)))
H_fixed[0, 44:56] = gain    # right
H_fixed[0, [4, 5, 16, 17]] = -gain   # left
H_fixed[1, 22:26] = gain    # up
H_fixed[1, 35:39] = -gain   # down

fixed_decoder = attach_extractor(train.make_fixed_wf_decoder(np.array(units), ssm, H_fixed, dt=binlen, n_taps=n_taps_fixed), units)
fixed_decoder.init_zscore(np.mean(neural_all, axis=0), np.std(neural_all, axis=0))  # or a constant guess, see above
plot_wf_weights(fixed_decoder, 'fixed')

In [ ]:
db.save_decoder(parent, fixed_decoder, 'fixed_wf')

## Check a saved decoder

Decoders saved above can be loaded back from a task entry that used them, or by name, to check what will run in the task.

In [ ]:
decoder = entry.get_decoder()  # the decoder used in this session, if any
print(type(decoder).__name__, 'binlen', decoder.binlen, 'extractor', decoder.extractor_cls.__name__, decoder.extractor_kwargs)
if hasattr(decoder.filt, 'n_taps'):
    print('n_taps', decoder.filt.n_taps, 'zscore', decoder.zscore, 'H', decoder.filt.H.shape)